#**🧠🔍 LangChain RAG Pipeline with FAISS + OpenAI + Llumo Evaluation**

###**📘 Notebook Overview**
This notebook implements a Retrieval-Augmented Generation (RAG) workflow using:

- 🔎 LangChain for document loading, splitting, and retrieval

- 🧠 OpenAI GPT (via ChatOpenAI) for answering user queries

- 📂 FAISS vector store for fast semantic search

- 🧪 Llumo SDK for evaluating generated outputs based on context usage, semantic quality, and hallucinations

It helps you **evaluate RAG context** using Llumo’s powerful context-level metrics as well as some additional metrics to ensure quality and safety:

✨ **Metrics included:**  
- **Context Utilization** ⚖️  
- **Redundancy Reduction** 🔍  
- **Relevance Retention** ☣️  
- **Semantic Cohesion**
- **Hallucination** 🚫  

---

###**📦 1. Install Required Packages**

In [5]:
# 📦 Install the Llumo SDK for evaluating  on multiple metrics like hallucination, context usage, etc.
!pip install llumo -q

# 🧠 Install LangChain — the core framework that helps us build RAG pipelines using retrievers, chains, and language models
!pip install langchain -q

# 🤖 Install the OpenAI Python SDK — needed to interact with OpenAI's GPT models like gpt-3.5-turbo or gpt-4
!pip install openai -q

# ⚡ Install FAISS CPU version — used as the vector store for fast similarity search (semantic retrieval of documents)
!pip install faiss-cpu -q

# 🔧 Install the updated LangChain Community package — this includes support for many document loaders, retrievers, and tools
!pip install -U langchain-community -q

# 📄 Install PyPDF — required to load and parse PDF documents using LangChain's PDF loaders
!pip install pypdf -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.9/438.9 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.1 MB/s eta 0:00:00


###**🔐 2. Setup API Keys**

In [3]:
import os

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = "Enter Your Open API Key"

# Set your Llumo API Key
os.environ["LLUMO_API_KEY"] = "Enter Your LLumo Key"

openai_key = os.getenv("OPENAI_API_KEY")
llumo_key = os.getenv("LLUMO_API_KEY")

###**3. Load and Split Data**

In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader

# Upload your PDF to Colab and replace the filename below
loader = PyPDFLoader("Howsuccessfulpeoplethink.pdf")  # Replace with your actual PDF filename
docs = loader.load()

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
split_docs = splitter.split_documents(docs)


###**🧬 4. Embed & Store in FAISS**

In [25]:
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

embedding = OpenAIEmbeddings(api_key=openai_key)
vectorstore = FAISS.from_documents(split_docs, embedding)


###**🤖 5. Setup OpenAI + RetrievalQA Chain**

In [26]:
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA

retriever = vectorstore.as_retriever( input_value={"k": 3})
llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo",api_key=openai_key)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type="stuff",

)


###**❓ 6. Ask Questions and Collect Results**


### **The data used for evaluation will be in the following Example format:**

- `query`: The input question  
- `context`: The contextual data retrieved from the database or other sources to assist in answering the query  
- `output`: The LLM final response as plain text


```
[  
  {
    "query": "What is the capital of France?",
    "context": "France is a country in Europe. Its capital city is Paris.",
    "output": "The capital of France is Paris."
  },
  {
    "query": "Summarize the plot of 'Romeo and Juliet'.",
    "context": 'Romeo and Juliet' is a tragedy by William Shakespeare.It is about two lovers from feuding families.",
    "output": "Romeo and Juliet is a tragedy by William Shakespeare about two young lovers whose deaths ultimately reconcile their feuding families."
  }
]
```



In [22]:
import json

# 🔍 List of input queries you want the RAG pipeline to answer
query_list = [
    "What are the 5 habits of successful people?",
    "How to think big?",
]

# 📦 This will store the final results in the required format: [{"query": ..., "context": ..., "output": ...}, ...]
results = []

# 🔁 Loop through each query and get the generated answer using RetrievalQA
for query in query_list:

    # 🎯 Send the query to the qa_chain (which handles both retrieval and LLM generation)
    response = qa_chain({"query": query})

    # ✅ Extract the generated answer from the response
    answer = response["result"]

    # 📚 Get the documents that were retrieved from the vector DB (FAISS) to use as context
    context_docs = response["source_documents"]

    # 🧩 Combine all the retrieved document text into one single string
    combined_context = "\n".join(doc.page_content for doc in context_docs)

    # 📝 Append the structured result to the results list
    results.append({
        "query": query,
        "context": combined_context,
        "output": answer
    })


###**📊 7. Evaluate with Llumo SDK**


In [23]:
# Import the evaluation client from Llumo SDK
from llumo import LlumoClient

# Initialize Llumo client
client = LlumoClient(api_key=llumo_key)

# Evaluate the RAG results
resultDf = client.evaluateMultiple(
    data = results,  # Input Data
    evals = ["Context Utilization", "Redundancy Reduction", "Relevance Retention","Semantic Cohesion","Hallucination"], # List of metrics
    createExperiment = False,   # Set to True to save results as an experiment on the Llumo platform. If False, returns results as a DataFrame or A Python Dict. - Optional
    getDataFrame = True, # Return result as a DataFrame (True) or dictionary (False) - Optional
)



Processing Batches: 100%|██████████| 5/5 [00:16<00:00,  3.33s/batch]


In [24]:
resultDf

,query,context,output,Context Utilization,Context Utilization Reason,Redundancy Reduction,Redundancy Reduction Reason,Relevance Retention,Relevance Retention Reason,Semantic Cohesion,Semantic Cohesion Reason,Hallucination,Hallucination Reason
0,What are the 5 habits of successFul people?,cultivate the \nhabit of doing so. The difficu...,"Based on the provided context, here are five s...",85,The response effectively uses the provided con...,53,The context contains some conceptual overlap a...,29,The context does not provide the 5 habits of s...,53,The context has some logical continuity but su...,37,The output includes minor details and vague in...
1,How to think big?,"different way, to break new ground, to find ne...","To think big, you can start by cultivating big...",84,The response effectively uses the provided con...,57,The context contains some redundancy. For exam...,86,The context provides relevant information on h...,94,The context maintains a good flow of informati...,29,The output includes minor details not explicit...
